# Análisis de Métricas de Error: MAPE y MASE

Comparación de los 3 modelos:
- TMT → SSRT
- TMT → K_6
- TMT → d'

In [ ]:
import pandas as pd
import numpy as np
import ast
import sys
from pathlib import Path

# Find project root (directory containing src/)
_p = Path().resolve()
while not (_p / 'src').is_dir() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

from src.config import REGRESSION_ANALYSIS_FOLDER

# Cargar consolidated
consolidated_dir = Path(REGRESSION_ANALYSIS_FOLDER) / "consolidated"
consolidated_path = sorted(consolidated_dir.glob("combined_summary_with_dispersion_*.csv"))[-1]
df_all = pd.read_csv(consolidated_path)
print(f"Loaded from: {consolidated_path.name}")

In [ ]:
# Cargar resultados desde consolidated
df_ssrt = df_all[(df_all["target"] == "ssrt") & (df_all["dataset"] == "tmt_ssrt")].reset_index(drop=True)
df_k6 = df_all[(df_all["target"] == "K_6") & (df_all["dataset"] == "tmt_k6")].reset_index(drop=True)
df_dprime = df_all[(df_all["target"] == "sensibilidad") & (df_all["dataset"] == "tmt_dprime")].reset_index(drop=True)

print("Datasets cargados:")
print(f"  - SSRT: {len(df_ssrt)} modelos")
print(f"  - K_6: {len(df_k6)} modelos")
print(f"  - d': {len(df_dprime)} modelos")

In [33]:
df_ssrt

,model,r2,mse,mae,y_true,y_pred
0,DummyRegressor,-0.011869,2696.308288,40.421441,"[235, 132, 205, 94, 175, 213, 255, 270, 201, 2...","[224.5680473372781, 225.17751479289942, 224.74..."
1,LinearRegression,-0.207797,3218.393023,45.279223,"[235, 132, 205, 94, 175, 213, 255, 270, 201, 2...","[277.4467398629047, 235.94749978816222, 246.88..."
2,Ridge,-0.011669,2695.773690,40.501880,"[235, 132, 205, 94, 175, 213, 255, 270, 201, 2...","[227.1066444973185, 225.9692969955806, 226.135..."
3,Lasso,-0.015558,2706.136094,40.511675,"[235, 132, 205, 94, 175, 213, 255, 270, 201, 2...","[224.5680473372781, 225.29743688324888, 224.74..."
4,XGBRegressor,-0.320542,3518.822682,46.661325,"[235, 132, 205, 94, 175, 213, 255, 270, 201, 2...","[263.0732727050781, 221.3926239013672, 210.432..."
5,ElasticNet,-0.012988,2699.290450,40.461915,"[235, 132, 205, 94, 175, 213, 255, 270, 201, 2...","[224.5680473372781, 225.17751479289942, 224.74..."
6,SVR,-0.012541,2698.097458,40.285499,"[235, 132, 205, 94, 175, 213, 255, 270, 201, 2...","[232.01157122586665, 231.2820123542773, 231.64..."
7,RandomForestRegressor,-0.104146,2942.194982,42.953788,"[235, 132, 205, 94, 175, 213, 255, 270, 201, 2...","[243.49798161685257, 229.42316859092355, 225.5..."


In [34]:
df_k6

,model,r2,mse,mae,y_true,y_pred
0,DummyRegressor,-0.010899,1.154102,0.848730,"[0.6, 2.8, 4.2, 2.2, 3.6, 0.8, 3.6, 1.53, 2.4,...","[2.4007608695652176, 2.388804347826087, 2.3811..."
1,LinearRegression,-0.116319,1.274456,0.917199,"[0.6, 2.8, 4.2, 2.2, 3.6, 0.8, 3.6, 1.53, 2.4,...","[1.9545476470260472, 2.314851872336263, 2.5901..."
2,Ridge,0.027146,1.110667,0.845062,"[0.6, 2.8, 4.2, 2.2, 3.6, 0.8, 3.6, 1.53, 2.4,...","[2.1099312703322104, 2.4299778540403247, 2.723..."
3,Lasso,-0.044597,1.192574,0.869554,"[0.6, 2.8, 4.2, 2.2, 3.6, 0.8, 3.6, 1.53, 2.4,...","[2.4007608695652176, 2.388804347826087, 2.3811..."
4,XGBRegressor,-0.019601,1.164037,0.891576,"[0.6, 2.8, 4.2, 2.2, 3.6, 0.8, 3.6, 1.53, 2.4,...","[1.9128085374832153, 2.0887765884399414, 2.666..."
5,ElasticNet,-0.056747,1.206445,0.889939,"[0.6, 2.8, 4.2, 2.2, 3.6, 0.8, 3.6, 1.53, 2.4,...","[2.4007608695652176, 2.4298125491591804, 2.727..."
6,SVR,-0.011006,1.154224,0.852428,"[0.6, 2.8, 4.2, 2.2, 3.6, 0.8, 3.6, 1.53, 2.4,...","[2.4581882402140853, 2.332919855729474, 2.6258..."
7,RandomForestRegressor,0.012825,1.127017,0.854177,"[0.6, 2.8, 4.2, 2.2, 3.6, 0.8, 3.6, 1.53, 2.4,...","[2.161549279609279, 2.3913718752803907, 2.8049..."


In [35]:
df_dprime

,model,r2,mse,mae,y_true,y_pred
0,DummyRegressor,-0.008869,2.753509,1.247817,"[-2.11, -0.84, 1.8, -2.09, 0.84, -1.23, 0.75, ...","[-0.0723008849557522, -0.07792035398230086, -0..."
1,LinearRegression,-0.050906,2.868240,1.288025,"[-2.11, -0.84, 1.8, -2.09, 0.84, -1.23, 0.75, ...","[0.2404236835894631, -0.3543190279784598, 0.04..."
2,Ridge,0.023378,2.665497,1.218940,"[-2.11, -0.84, 1.8, -2.09, 0.84, -1.23, 0.75, ...","[-0.5009623642246178, 0.0019365418732197187, 0..."
3,Lasso,0.001427,2.725409,1.235018,"[-2.11, -0.84, 1.8, -2.09, 0.84, -1.23, 0.75, ...","[-0.5313686696830904, 0.01957184892927341, 0.3..."
4,XGBRegressor,-0.201483,3.279211,1.360194,"[-2.11, -0.84, 1.8, -2.09, 0.84, -1.23, 0.75, ...","[-0.6612340211868286, -0.3610857427120209, 0.4..."
5,ElasticNet,0.017931,2.680365,1.225556,"[-2.11, -0.84, 1.8, -2.09, 0.84, -1.23, 0.75, ...","[-0.5985906373867917, 0.0032375375235073195, 0..."
6,SVR,-0.008000,2.751138,1.238495,"[-2.11, -0.84, 1.8, -2.09, 0.84, -1.23, 0.75, ...","[-0.9158682339605095, 0.25775122871109035, 0.6..."
7,RandomForestRegressor,0.004053,2.718241,1.234420,"[-2.11, -0.84, 1.8, -2.09, 0.84, -1.23, 0.75, ...","[-0.5935397222469896, -0.2400640646865877, 0.5..."


In [36]:
from sklearn.metrics import mean_absolute_percentage_error

def add_mape(df):
    """Calcula MAPE para cada modelo y lo agrega como columna."""
    mape_values = []
    for _, row in df.iterrows():
        y_true = np.array(ast.literal_eval(row['y_true']))
        y_pred = np.array(ast.literal_eval(row['y_pred']))
        mape = mean_absolute_percentage_error(y_true, y_pred) * 100
        mape_values.append(mape)
    df['mape'] = mape_values
    return df

In [37]:
df_ssrt = add_mape(df_ssrt)
df_ssrt[['model', 'r2', 'mae', 'mape']].sort_values('mape')

,model,r2,mae,mape
0,DummyRegressor,-0.011869,40.421441,22.324023
5,ElasticNet,-0.012988,40.461915,22.340773
2,Ridge,-0.011669,40.501880,22.354937
3,Lasso,-0.015558,40.511675,22.370686
6,SVR,-0.012541,40.285499,22.685063
7,RandomForestRegressor,-0.104146,42.953788,23.746826
1,LinearRegression,-0.207797,45.279223,24.784025
4,XGBRegressor,-0.320542,46.661325,25.600517


In [40]:
def smape(y_true, y_pred):
    """Symmetric MAPE - más robusto con valores cercanos a cero."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator != 0
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denominator[mask]) * 100

def add_smape(df):
    smape_values = []
    for _, row in df.iterrows():
        y_true = np.array(ast.literal_eval(row['y_true']))
        y_pred = np.array(ast.literal_eval(row['y_pred']))
        smape_values.append(smape(y_true, y_pred))
    df['smape'] = smape_values
    return df

In [ ]:
df_k6 = add_mape(df_k6)
df_k6 = add_smape(df_k6)
df_k6[['model', 'r2', 'mae', 'mape', 'smape']].sort_values('smape')

In [ ]:
df_dprime = add_mape(df_dprime)
df_dprime = add_smape(df_dprime)
df_dprime[['model', 'r2', 'mae', 'mape', 'smape']].sort_values('smape')